In [3]:
import pandas as pd
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

#Training the model
train_path = "KDDTrain+[1].txt"

columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in",
    "num_compromised","root_shell","su_attempted","num_root","num_file_creations",
    "num_shells","num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
    "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate",
    "srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
    "dst_host_serror_rate","dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","label","difficulty"
]

df = pd.read_csv(train_path, header=None, names=columns)
df.drop("difficulty", axis=1, inplace=True)

cat_cols = ["protocol_type", "service", "flag"]
encoders = {}

#Encode categorical fields
for c in cat_cols:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c])
    encoders[c] = le

#Encoding labels 
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["label"])

X_train = df.drop("label", axis=1)
y_train = df["label"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

model = RandomForestClassifier(n_estimators=150, max_depth=20, random_state=42)
model.fit(X_train_scaled, y_train)

print("Model Ready!")

#GUI
prediction_results = None
uploaded_file_path = None

def transform_val(le, values):
    known = set(le.classes_)
    return [le.transform([v])[0] if v in known else 0 for v in values]

def upload_csv():
    global prediction_results, uploaded_file_path

    file = filedialog.askopenfilename(
        title="Select CSV File", filetypes=[("CSV Files", "*.csv")]
    )
    if not file:
        return

    uploaded_file_path = file

    try:
        test_df = pd.read_csv(file)
        test_df = test_df[X_train.columns]
        #Validate columns
        if list(test_df.columns) != list(X_train.columns):
            messagebox.showerror("Error", "CSV must have exactly the 41 NSL-KDD feature columns.")
            return

        #Transform categorical
        for c in cat_cols:
            test_df[c] = transform_val(encoders[c], test_df[c])

        #Scale features
        X_scaled = scaler.transform(test_df)

        #Prediction: convert numbers back to attack names
        preds = label_encoder.inverse_transform(model.predict(X_scaled))

        test_df["Prediction"] = preds
        prediction_results = test_df

        #Output
        results_box.config(state="normal")
        results_box.delete("1.0", tk.END)
        results_box.insert(tk.END, f"✔ Rows classified: {len(preds)}\n\n")
        results_box.insert(tk.END, f"✔ Detected classes: {test_df['Prediction'].unique().tolist()}\n\n")
        results_box.insert(tk.END, "🔍 Sample predictions:\n")
        for i in range(min(10, len(preds))):
            results_box.insert(tk.END, f"  ➤ Row {i+1}: {preds[i]}\n")
        results_box.config(state="disabled")

        messagebox.showinfo("Done", "Classification Complete!")

    except Exception as e:
        messagebox.showerror("Error", str(e))


def save_results():
    if prediction_results is None:
        messagebox.showerror("Error", "No predictions to save.")
        return

    if uploaded_file_path is None:
        messagebox.showerror("Error", "No input file loaded.")
        return

    try:
        prediction_results.to_csv(uploaded_file_path, index=False)
        messagebox.showinfo("Saved", f"Predictions saved to:\n{uploaded_file_path}")
    except Exception as e:
        messagebox.showerror("Error", str(e))

#Colorful user interface
root = tk.Tk()
root.title("Intrusion Detection System - NSL-KDD")
root.geometry("780x620")
root.configure(bg="#E8ECF8")

#Gradient Banner
header = tk.Canvas(root, height=90, bg="#E8ECF8", highlightthickness=0)
header.pack(fill="x")

for i in range(90):
    color = f"#%02x%02x%02x" % (230 - i//3, 210 - i//6, 255)
    header.create_rectangle(0, i, 800, i+1, outline="", fill=color)

header.create_text(
    390, 45,
    text="🌐 Network Intrusion Detection System",
    font=("Century Gothic", 22, "bold"),
    fill="#2D006E"
)

#Button theme
style = ttk.Style()
style.theme_use("clam")

style.configure(
    "Pretty.TButton",
    font=("Segoe UI", 12, "bold"),
    padding=10,
    foreground="#FFFFFF",
    background="#5A3EEC",
    bordercolor="#5A3EEC",
    focusthickness=3,
    focuscolor="#A18CFF"
)

style.map(
    "Pretty.TButton",
    foreground=[("active", "#FFFFFF")],
    background=[("active", "#7154FF"), ("pressed", "#512EE2")],
)

#Buttons
btn_frame = tk.Frame(root, bg="#E8ECF8")
btn_frame.pack(pady=20)

upload_btn = ttk.Button(btn_frame, text="📂 Upload CSV & Classify", style="Pretty.TButton", command=upload_csv)
upload_btn.grid(row=0, column=0, padx=10)

save_btn = ttk.Button(btn_frame, text="💾 Save Results", style="Pretty.TButton", command=save_results)
save_btn.grid(row=0, column=1, padx=10)

#Results Panel
panel = tk.Frame(root, bg="#FFFFFF")
panel.pack(padx=30, pady=20, fill="both", expand=True)

panel_title = tk.Label(
    panel,
    text="📊 Classification Output",
    bg="#FFFFFF",
    fg="#4733A3",
    font=("Century Gothic", 16, "bold")
)
panel_title.pack(pady=10)

results_box = tk.Text(
    panel,
    height=20,
    width=80,
    font=("Consolas", 12),
    bg="#F4F1FF",
    fg="#2A134F",
    bd=0,
    wrap="word"
)
results_box.pack(padx=20, pady=10)
results_box.config(state="disabled")

root.mainloop()


Model Ready!
